# W11A · 8B · 실제 이력에서 다음 아이템 추천하기

**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026-11-10**

예상 → 계산 → 실행 → 한 조건 수정 → 이유 설명 순서로 기록하세요. 자신의 Colab 사본을 저장하고 첫 셀부터 실행합니다. 기본값은 CPU이며 공식 데이터 자동 준비·캐시 재사용을 지원합니다. 설치 갱신 안내가 나오면 런타임 재시작 뒤 처음부터 실행하세요.

`None`은 자신의 답을 적는 자리이며 예제 실행 변수와 분리돼 있습니다. 예제가 끝까지 실행됐다고 자신의 문제 풀이가 완료된 것은 아닙니다.


In [ ]:
from __future__ import annotations
import os, sys, subprocess
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_VERSION = '2026-fall-w11a'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        f'luna-recommender-course[apps,neural] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.31.0'], check=True)
    for package, wanted in [('tensorflow','2.21.0'), ('keras','3.15.1')]:
        if package in sys.modules and getattr(sys.modules[package], '__version__', None) != wanted:
            raise RuntimeError('설치가 갱신되었습니다. 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요.')
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from IPython.display import display
from luna_recsys.autoencoder_recommenders import (
    toy_ratings, fixed_ae_forward, masked_metrics, loss_comparison,
    build_fixed_autoencoder, prepare_autoencoder_data, oriented_arrays,
    fit_autoencoder, predict_autoencoder, evaluate_autoencoder, recommend_autoencoder,
)
from luna_recsys.autoencoder_explainer import (
    ae_view, export_ae_csv, build_autoencoder_explainer,
)
from luna_recsys.datasets import prepare_movielens
print('Python',sys.version.split()[0], 'NumPy',np.__version__, 'TensorFlow',tf.__version__, 'Keras',keras.__version__)
print('수업 ref:',COURSE_VERSION,'CPU 실행; 학습 자료 모드를 결과와 함께 확인합니다.')

from typing import Any, Callable


In [ ]:
from luna_recsys.sequential_recommenders import (
    scaled_attention, pad_history, prefix_pairs, prepare_sequence_data,
    build_sequence_model, sequence_train_step, fit_sequence,
    select_candidates, rollout, ranking_metrics, evaluate_sequence,
)
from luna_recsys.transformer_explainer import (
    attention_view, export_attention, sequence_view, export_sequence, build_transformer_explainer,
)


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/05-prefix-pairs.png" alt="접두 이력과 다음 정답" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/05-prefix-pairs.png)

독자 계산 도해 또는 교재 개념 재구성입니다. 강의노트의 그림별 출처와 설명을 함께 읽으세요.


### 本문 실행 정의 · `pad_history`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: a7d2223b908d8578997f08aedc3e292d26b1baea443e07142617d115fa4e1773
def pad_history(history: Any, maxlen: int) -> np.ndarray:
    """Right-pad/truncate the most recent positive internal IDs. New int32 array."""
    ids = np.asarray(history, dtype=np.int32)
    if ids.ndim != 1 or not len(ids) or np.any(ids <= 0) or maxlen < 1:
        raise ValueError('A nonempty positive-ID history and maxlen>=1 are required.')
    out = np.zeros(maxlen, dtype=np.int32)
    tail = ids[-maxlen:]
    out[:len(tail)] = tail
    return out


### 本문 실행 정의 · `prefix_pairs`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: 1f0311ae11eff9b9aed0b0be7a51bebdcb1157cf0a498436eaeb5c7f2dc054b7
def prefix_pairs(history: Any, maxlen: int = 20, min_prefix: int = 20) -> tuple:
    """Observed prefix -> next label; never includes that label in its input.

    A history of length n yields max(0,n-min_prefix) rows. No shuffle or learning.
    """
    ids = list(history)
    if min_prefix < 1:
        raise ValueError('min_prefix must be positive.')
    xs = [pad_history(ids[:end], maxlen) for end in range(min_prefix, len(ids))]
    ys = ids[min_prefix:]
    return np.asarray(xs, dtype=np.int32).reshape(-1, maxlen), np.asarray(ys, dtype=np.int32)


## 공식 자료 준비와 사용자 분리

`mode=real`은 실제 자료를 사용하며 실패를 합성 자료로 바꾸지 않습니다. 사용자 분리 후 접두 쌍을 만들며, 본문 `prefix_pairs`를 pair_fn으로 전달합니다. 기본은 적격 사용자 최대120명/훈련쌍1500개/최근20개 이력입니다. tied timestamp의 파일 행 순서는 실제 시청 순서를 보장하지 않습니다.


In [ ]:
prepared=prepare_movielens(Path.home()/'.cache/luna-recsys',mode='real')
data=prepare_sequence_data(prepared,maxlen=20,max_users=120,max_train_pairs=1500,pair_fn=prefix_pairs)
print('자료 모드:',prepared.mode,prepared.description)
print('사용자 분리:',{k:len(v) for k,v in data.users.items()})
print('입력/정답:',{k:(data.x[k].shape,data.y[k].shape) for k in data.x})
print('실험 조건:',data.protocol)
print('첫 입력:',data.x['train'][0], '다음 정답 내부ID:',data.y['train'][0])


## T05 · 직접 설명하고 수정하기

[1,2,3,4]에서 최소 접두2·maxlen3의 입력/정답 두 쌍을 적으세요. ID 0과 실제 정답의 역할, 증강 전 사용자 분리 이유를 설명하세요.


In [ ]:
my_t05 = None


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/11-encoder-path.png" alt="임베딩에서 분류까지" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/11-encoder-path.png)

독자 계산 도해 또는 교재 개념 재구성입니다. 강의노트의 그림별 출처와 설명을 함께 읽으세요.


### 本문 실행 정의 · `build_sequence_model`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: a47d38ad60f404d9c81a4c3922940fdcdcaf74b4d35284491f48fb1b6fd33468
def build_sequence_model(vocabulary: int, *, maxlen: int = 20, embed_dim: int = 16,
                         heads: int = 2, key_dim: int = 8, ff_dim: int = 32,
                         blocks: int = 1, dense_dim: int = 64, dropout: float = .1,
                         seed: int = 20261110) -> tuple:
    """Item+position -> encoder(s) -> Flatten/Dense -> next-item softmax.

    Return (classifier, first-block attention probe). Explicit key padding mask
    and post-block zeroing avoid relying on custom-layer mask propagation.
    Prefix-to-one-label task permits attention throughout the observed prefix.
    key_dim is per head; output width returns to embed_dim via projection.
    """
    import tensorflow as tf
    import keras
    if min(vocabulary, maxlen, embed_dim, heads, key_dim, ff_dim, blocks, dense_dim) < 1:
        raise ValueError('All model dimensions must be positive.')
    keras.utils.set_random_seed(seed)
    ids = keras.Input((maxlen,), dtype='int32', name='internal_item_ids')
    valid = keras.layers.Lambda(lambda z: tf.not_equal(z, 0), name='valid_items')(ids)
    positions = keras.layers.Lambda(lambda z: tf.broadcast_to(tf.range(maxlen)[None, :], tf.shape(z)), name='positions')(ids)
    x = keras.layers.Embedding(vocabulary, embed_dim, name='item_embedding')(ids)
    pos = keras.layers.Embedding(maxlen, embed_dim, name='position_embedding')(positions)
    x = keras.layers.Add()([x, pos])
    multiplier = keras.layers.Lambda(lambda z: tf.cast(z[..., None], tf.float32), name='pad_multiplier')(valid)
    x = keras.layers.Multiply()([x, multiplier])
    attention_mask = keras.layers.Lambda(lambda z: z[:, None, :], name='key_padding_mask')(valid)
    first_scores = None
    for j in range(blocks):
        attention = keras.layers.MultiHeadAttention(num_heads=heads, key_dim=key_dim, name=f'attention_{j}')
        attended, scores = attention(x, x, attention_mask=attention_mask, return_attention_scores=True)
        if first_scores is None:
            first_scores = scores
        attended = keras.layers.Dropout(dropout)(attended)
        x = keras.layers.LayerNormalization(epsilon=1e-6)(keras.layers.Add()([x, attended]))
        f = keras.layers.Dense(ff_dim, activation='relu')(x)
        f = keras.layers.Dense(embed_dim)(f)
        f = keras.layers.Dropout(dropout)(f)
        x = keras.layers.LayerNormalization(epsilon=1e-6)(keras.layers.Add()([x, f]))
        x = keras.layers.Multiply()([x, multiplier])
    flat = keras.layers.Flatten()(x)
    hidden = keras.layers.Dense(dense_dim, activation='relu')(flat)
    probs = keras.layers.Dense(vocabulary, activation='softmax', name='next_item_probabilities')(hidden)
    return keras.Model(ids, probs), keras.Model(ids, first_scores)


### 本문 실행 정의 · `sequence_train_step`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: fb8ad2733fc3a47bc0202cc46e019f3acfaee068d57584d417bf8c549a37674b
def sequence_train_step(model: Any, optimizer: Any, x: Any, y: Any, loss_fn: Callable = None) -> dict:
    """One real update; integer labels(B,), probs(B,V), CE mean over rows.

    loss_fn(y,probabilities) may be supplied from an editable notebook cell.
    Changes weights/optimizer state. Returns scalar loss and gradient norm.
    """
    import tensorflow as tf
    import keras
    loss_fn = loss_fn or keras.losses.SparseCategoricalCrossentropy()
    with tf.GradientTape() as tape:
        probs = model(x, training=True)
        loss = tf.reduce_mean(loss_fn(y, probs))
        if model.losses:
            loss = loss + tf.add_n(model.losses)
    grads = tape.gradient(loss, model.trainable_variables)
    if not bool(tf.math.is_finite(loss)) or any(g is None or not bool(tf.reduce_all(tf.math.is_finite(tf.convert_to_tensor(g)))) for g in grads):
        raise FloatingPointError('Nonfinite loss/gradient; no fallback.')
    norm = float(tf.linalg.global_norm(grads))
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return {'loss': float(loss), 'gradient_norm': norm}


## 손실 계산과 실제 갱신 한 번

본문 모델·학습 함수를 사용해 입력2개를 학습합니다. 이 진단 모형은 뒤의 본 실험과 구분합니다. 학습률·모수 변화·기울기 크기를 확인합니다.


In [ ]:
diagnostic_model,_=build_sequence_model(len(data.encode)+1,maxlen=data.maxlen)
diagnostic_before=[w.copy() for w in diagnostic_model.get_weights()]
diagnostic_update=sequence_train_step(diagnostic_model,keras.optimizers.Adam(.0003),data.x['train'][:2],data.y['train'][:2])
diagnostic_delta=sum(float(np.sum(abs(a-b))) for a,b in zip(diagnostic_before,diagnostic_model.get_weights()))
print('한 번의 실제 갱신:',diagnostic_update,'모수 변화:',diagnostic_delta)
assert diagnostic_delta>0


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/12-cross-entropy.png" alt="정답 확률과 손실" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/12-cross-entropy.png)

독자 계산 도해 또는 교재 개념 재구성입니다. 강의노트의 그림별 출처와 설명을 함께 읽으세요.


## T06 · 직접 설명하고 수정하기

확률 [.2,.5,.3]에서 정답 두 번째의 CE를 계산하세요. 순전파만 실행할 때와 sequence_train_step 실행 때 어떤 상태가 달라지나요?


In [ ]:
my_t06 = None


## 작은 CPU 실험과 검증 기준 선택

임베딩16/헤드2/key_dim8/블록1/FFN32/Dense64, Adam .0003, 최대3epoch입니다. 실제 데이터의 축소 실험이며 교재 전체 훈련의 재현이 아닙니다. 수정한 본문 함수가 model_fn과 step_fn으로 실제 훈련에 전달됩니다.


In [ ]:
fitted=fit_sequence(data,epochs=3,batch_size=64,model_fn=build_sequence_model,step_fn=sequence_train_step)
display(fitted.history)
print('설정:',fitted.config,'갱신 수:',fitted.updates,'초:',fitted.seconds,'선택 epoch:',fitted.best_epoch)
padded_attention=fitted.probe(data.x['validation'][:1],training=False).numpy()
print('attention shape:',padded_attention.shape)
from matplotlib.figure import Figure
learning_figure=Figure(figsize=(10,3.6))
learning_axes=learning_figure.subplots(1,2)
learning_axes[0].plot(fitted.history.epoch,fitted.history.train_ce,'o-',label='train CE')
learning_axes[0].plot(fitted.history.epoch,fitted.history.validation_ce,'o-',label='validation CE')
learning_axes[0].set(xlabel='Epoch',ylabel='CE',title='Select using validation only');learning_axes[0].legend()
learning_axes[1].imshow(padded_attention[0,0],cmap='Blues',aspect='auto')
learning_axes[1].set(xlabel='Key position',ylabel='Query position',title='Learned head 1: given validation prefix')
learning_figure.tight_layout();display(learning_figure)


<img src="https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/13-own-rollout.png" alt="자기 예측만 추가하는 반복 생성" width="1100">

[그림 원본](https://raw.githubusercontent.com/lunalab-ai/recommender/2026-fall-w11a/course/notion/assets/w11a/13-own-rollout.png)

독자 계산 도해 또는 교재 개념 재구성입니다. 강의노트의 그림별 출처와 설명을 함께 읽으세요.


### 本문 실행 정의 · `select_candidates`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: 12ee3623104ebaee55bb9e43014c692ad86f8074b4dbba9b0d7bf9cc273e9abf
def select_candidates(scores: Any, history: Any, k: int = 1, *, exclude_seen: bool = True) -> list:
    """Rank internal IDs; exclude0/nonfinite and optionally already observed IDs.

    Stable ID-order tie break. No target-label access; does not mutate scores.
    """
    scores = np.asarray(scores, dtype=float).copy()
    if scores.ndim != 1 or k < 1:
        raise ValueError('A score vector and k>=1 are needed.')
    permitted = np.isfinite(scores)
    permitted[0] = False
    if exclude_seen:
        seen = np.asarray(list(history), dtype=int)
        if len(seen) and (seen.min() < 1 or seen.max() >= len(scores)):
            raise ValueError('History must contain internal vocabulary IDs.')
        permitted[seen] = False
    ids = np.flatnonzero(permitted)
    return ids[np.argsort(-scores[ids], kind='stable')[:k]].tolist()


### 本문 실행 정의 · `rollout`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: 7b1717738409786a49e1feb54f2c1156d5f8c5ac6013b58aeef22f9fe1ec7ee8
def rollout(score_fn: Callable, history: Any, *, maxlen: int, n: int = 7,
            candidate_fn: Callable = select_candidates) -> list:
    """Append own predicted INTERNAL IDs to a copied history, never future truth.

    score_fn receives one padded array(1,T) and returns(1,V) or(V,).
    candidate_fn(scores,history,k) controls actual later recommendation/app output.
    """
    current = list(history)
    if n < 1:
        raise ValueError('n must be positive.')
    chosen = []
    for _ in range(n):
        scores = np.asarray(score_fn(pad_history(current, maxlen)[None, :])).reshape(-1)
        ids = candidate_fn(scores, current, 1)
        if not ids:
            raise ValueError('No eligible candidates remain.')
        item = int(ids[0])
        if item <= 0 or item >= len(scores):
            raise ValueError('Candidate is not a valid internal ID.')
        chosen.append(item)
        current.append(item)
    return chosen


### 本문 실행 정의 · `ranking_metrics`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: f141613022a55dd1714e9c09c7caad482e6811011a30bb5d0bafaccc8e30943c
def ranking_metrics(recommended: Any, target: Any) -> dict:
    """Set hits / returned-list length and unique held-out targets; not positional accuracy."""
    rec, truth = list(recommended), set(target)
    if not rec or not truth or len(set(rec)) != len(rec) or 0 in rec:
        raise ValueError('A nonempty unique recommendation and nonempty target set are required.')
    hits = len(set(rec) & truth)
    precision, recall = hits/len(rec), hits/len(truth)
    return {'hits': hits, 'precision': precision, 'recall': recall,
            'f1': 2*precision*recall/(precision+recall) if hits else 0.}


### 本문 실행 정의 · `evaluate_sequence`

아래 정의를 먼저 읽고 실행합니다. 준비 코드는 모듈에 두고 핵심 계산은 이 셀에서 수정합니다. 초기 정의는 공통 모듈에서 추출했습니다. 수정 후 이 셀과 후속 계산 셀을 다시 실행하세요.


In [ ]:
# Initial source SHA-256: bb0293fc397ddbfd3e62c7d9a03f3e13547e0c1f66fec2f8b74458a68307ff8b
def evaluate_sequence(data: SequenceData, score_fn: Callable, *, n: int = 7,
                      candidate_fn: Callable = select_candidates) -> pd.DataFrame:
    """Each test user's own history/targets; return per-user metrics and equal-user macro row externally."""
    rows = []
    for user in data.users['test']:
        rec = rollout(score_fn, data.histories[user], maxlen=data.maxlen, n=n, candidate_fn=candidate_fn)
        rows.append({'user_id': user, **ranking_metrics(rec, data.targets[user])})
    return pd.DataFrame(rows)


## 같은 프로토콜의 실제 평가와 인기 기준선

모형은 자체 예측을 입력에 추가해7개를 생성합니다. 각 사용자의 보류10개와 집합으로 비교합니다. 인기 기준선은 훈련 사용자만 사용하며 동일한 후보 규칙을 따릅니다. 이 결과로 다시 설정을 고르지 않습니다.


In [ ]:
score_fn=lambda x:fitted.model(x,training=False).numpy()
model_scores=evaluate_sequence(data,score_fn,candidate_fn=select_candidates)
popularity_scores=evaluate_sequence(data,lambda x:data.popularity,candidate_fn=select_candidates)
comparison=pd.DataFrame([{'model':'training popularity',**popularity_scores[['precision','recall','f1']].mean().to_dict()},
                         {'model':'small Transformer',**model_scores[['precision','recall','f1']].mean().to_dict()}])
display(comparison)
selected_user=data.users['test'][0]
message,recommendations=sequence_view(data,fitted,selected_user,candidate_fn=select_candidates)
print(message);display(recommendations)


## T07 · 직접 설명하고 수정하기

추천7개 중 보류 정답10개와2개가 겹칠 때 P/R/F1을 구하세요. 반복 생성 중 실제 다음 정답을 넣으면 왜 다른 실험이 되는지 설명하세요.


In [ ]:
my_t07 = None


## 함수 수정과 누적 앱

`candidate_fn`이 본문 후보 함수입니다. 한 규칙만 수정하고 이 셀을 다시 실행하면 추천표·CSV·앱에 같은 규칙을 적용합니다. 미래 정답을 후보 선택에 사용하지 마세요. 앱은 앞서 학습한 모형을 재사용합니다.


In [ ]:
csv_path=export_sequence(data,fitted,selected_user,candidate_fn=select_candidates)
assert pd.read_csv(csv_path).internal_id.tolist()==recommendations.internal_id.tolist()
demo=build_transformer_explainer(data=data,fitted=fitted,candidate_fn=select_candidates)
if IN_COLAB:
    demo.queue(default_concurrency_limit=1).launch(share=True,inline=False,prevent_thread_lock=True)
else:
    print('앱 객체/콜백 확인; 브라우저 실제 조작은 별도 검증')


## T08 · 직접 설명하고 수정하기

연습8-1: 길이·헤드·학습률 중 하나만 바꾸는 비교 실험을 설계하세요. 앱 함수 수정→표→CSV의 연결 인수도 찾고, RMSE와 CE/P@7을 직접 비교할 수 없는 이유를 적으세요.


In [ ]:
my_t08 = None


In [ ]:
assert prepared.mode in {'local','downloaded','cache','movielens'} or 'synthetic' not in prepared.mode
assert fitted.updates>0 and diagnostic_delta>0
assert len(recommendations)==7 and recommendations.internal_id.is_unique
assert (recommendations.internal_id>0).all()
assert set(data.users['train']).isdisjoint(data.users['validation'])
assert set(data.users['test']).isdisjoint(data.users['train']+data.users['validation'])
assert np.isfinite(comparison[['precision','recall','f1']].to_numpy()).all()
assert not set(recommendations.internal_id)&set(data.histories[selected_user])
print('W11A_8B_RUNTIME_PASS: real CPU fit, own-user evaluation, callback/CSV; learner responses separate')


## 점검 퀴즈

1. 7장의 평점 복원과 8장의 다음 아이템 예측은 입력·정답·손실이 어떻게 다른가요?
2. 중심화된 관측값이 0일 때도 손실에 포함해야 하는 이유는 무엇인가요?
3. Q가 3×2, K가 3×2, V가 3×2이면 attention 가중치와 출력의 크기는 각각 무엇인가요?
4. attention 가중치의 한 행 합이 1인 이유와 V에 곱하는 이유는 무엇인가요?
5. 패딩 마스크와 인과 마스크는 무엇을 다르게 가리나요?
6. 정답 아이템 확률이 .5일 때 교차엔트로피는 얼마이며, 학습은 무엇을 바꾸나요?
7. 반복 추천에서 원본 ID와 내부 ID, 실제 다음 정답은 각각 어디에 사용하나요?
8. 추천 7개 중 정답 10개와 겹치는 아이템이 2개일 때 P/R/F1을 구하고 평가 의미를 설명하세요.


## 기억할 것과 참고자료

입력·정답·중간값·실제 갱신·평가를 연결하세요.

- 주교재 임일(2025), 8장 전체 pp.158–176, 연습8-1. 독자 수치와 보완 구현을 구분합니다.
- [Attention 원 논문](https://arxiv.org/html/1706.03762v7)
- [Keras MultiHeadAttention](https://keras.io/api/layers/attention_layers/multi_head_attention/)
- [MovieLens 이용 조건](https://files.grouplens.org/datasets/movielens/ml-100k-README.txt)
